<a href="https://colab.research.google.com/github/tsheznick-sys/Shadow-Operation/blob/main/84546f26-shadow-operation/backend/Shadow_Operation_Backend_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""# %% [markdown]
# # Shadow-Operation — Backend Infrastructure (Colab Edition)
#
# **Full-stack influencer marketing platform backend** — runs entirely in Google Colab.
#
# This notebook delivers:
# 1. **Database Schema** — 7 tables, 30+ indexes, SQLite (swap to PostgreSQL for prod)
# 2. **Meta Graph API Client** — async, batch-capable, rate-limited
# 3. **Bulk Update Engine** — sync 1,000 creators in ~20 API calls
# 4. **REST API** — FastAPI with 9 endpoints (live preview in Colab)
# 5. **Meta API Auth Guide** — complete setup walkthrough
#
# > **To run**: Execute cells top-to-bottom. No external services needed.
# """


'# %% [markdown]\n# # Shadow-Operation — Backend Infrastructure (Colab Edition)\n#\n# **Full-stack influencer marketing platform backend** — runs entirely in Google Colab.\n#\n# This notebook delivers:\n# 1. **Database Schema** — 7 tables, 30+ indexes, SQLite (swap to PostgreSQL for prod)\n# 2. **Meta Graph API Client** — async, batch-capable, rate-limited\n# 3. **Bulk Update Engine** — sync 1,000 creators in ~20 API calls\n# 4. **REST API** — FastAPI with 9 endpoints (live preview in Colab)\n# 5. **Meta API Auth Guide** — complete setup walkthrough\n#\n# > **To run**: Execute cells top-to-bottom. No external services needed.\n# '

## 0 · Install Dependencies


In [2]:
!pip install -q sqlalchemy aiohttp fastapi uvicorn pydantic python-dotenv nest_asyncio httpx


In [3]:
import os, sys, json, csv, uuid, asyncio, time, logging, hashlib, hmac
from datetime import datetime, timezone, timedelta
from typing import Optional, Any
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from pathlib import Path
from collections import Counter

import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in Colab

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("shadow_op")

print("✅ Dependencies loaded")



✅ Dependencies loaded


---
## 1 · Database Schema

Seven tables covering the full creator lifecycle:

| Table | Records | Purpose |
|-------|---------|---------|
| `creators` | ~1,500 | Central entity — metrics, scoring, pipeline status |
| `creator_contacts` | ~128 | Email addresses with source tracking |
| `engagement_snapshots` | time-series | Daily metric captures for trend analysis |
| `assessments` | per-run | Scoring audit trail with model versioning |
| `outreach_events` | per-touch | Every email sent, DM, reply, call |
| `campaigns` | product-level | Offer grouping with revenue splits |
| `sync_log` | per-run | Meta API call audit for compliance |

**Indexed for**: handle lookup, niche+status+score filtering, full-text search,
demographic cross-filtering (ethnicity, location), engagement ranges, and
follower windows (5K–500K).


In [4]:
from sqlalchemy import (
    create_engine, Column, Integer, BigInteger, String, Text, Float, Boolean,
    DateTime, ForeignKey, Index, UniqueConstraint, CheckConstraint, func,
    event, Enum as SAEnum, text as satext
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.pool import StaticPool


# ── Base ─────────────────────────────────────────────────────────────────

class Base(DeclarativeBase):
    __abstract__ = True
    created_at = Column(DateTime(timezone=True), server_default=func.now(), nullable=False)
    updated_at = Column(DateTime(timezone=True), server_default=func.now(),
                        onupdate=func.now(), nullable=False)


# ── Enums (stored as strings for SQLite compat) ──────────────────────────

import enum

class CreatorStatus(str, enum.Enum):
    NEW_DISCOVERY = "NEW_DISCOVERY"
    DISCOVERED = "DISCOVERED"
    ASSESSED = "ASSESSED"
    QUALIFIED = "QUALIFIED"
    DRAFT_STAGED = "DRAFT_STAGED"
    OUTREACH_SENT = "OUTREACH_SENT"
    REPLIED = "REPLIED"
    WARM = "WARM"
    CALL_SCHEDULED = "CALL_SCHEDULED"
    CONTRACTED = "CONTRACTED"
    LIVE = "LIVE"
    COMPLETED = "COMPLETED"
    DECLINED = "DECLINED"
    QUARANTINED = "QUARANTINED"
    COLD = "COLD"
    HOLD = "HOLD"

class NicheCategory(str, enum.Enum):
    VIDEO_EDITING = "Video Editing"
    MOTION_GRAPHICS = "Motion Graphics"
    VFX = "VFX"
    THREE_D_BLENDER = "3D / Blender"
    ANIMATION = "Animation"
    DIGITAL_ART = "Digital Art"
    GRAPHIC_DESIGN = "Graphic Design"
    AI_VIDEO = "AI Video"
    FILMMAKING = "Filmmaking"
    PHOTOGRAPHY = "Photography"
    MUSIC_DJ = "Music / DJ"
    COLOR_GRADING = "Color Grading"
    ILLUSTRATION = "Illustration"
    OTHER = "Other"

class GateResult(str, enum.Enum):
    GREEN = "GREEN"
    AMBER = "AMBER"
    REJECT = "REJECT"

class Tier(str, enum.Enum):
    P0 = "P0"; P1 = "P1"; P2 = "P2"; P3 = "P3"

class EthnicityCategory(str, enum.Enum):
    WHITE = "White"
    BLACK = "Black"
    HISPANIC_LATINO = "Hispanic/Latino"
    ASIAN = "Asian"
    SOUTH_ASIAN = "South Asian"
    MIDDLE_EASTERN = "Middle Eastern"
    NATIVE_AMERICAN = "Native American"
    PACIFIC_ISLANDER = "Pacific Islander"
    MIXED = "Mixed"
    OTHER = "Other"
    PREFER_NOT_TO_SAY = "Prefer Not to Say"


# ── Table: creators ─────────────────────────────────────────────────────

class Creator(Base):
    """
    Central entity — one row per unique creator.
    30+ indexes designed for primary query patterns:
      • Lookup by handle (unique, case-insensitive)
      • Lookup by instagram_id (Meta canonical)
      • Filter by niche + status + score range
      • Filter by location, ethnicity, tier
      • Full-text search on handle and notes
    """
    __tablename__ = "creators"

    id            = Column(Integer, primary_key=True, autoincrement=True)
    creator_uid   = Column(String(36), unique=True, nullable=False, index=True)
    instagram_id  = Column(String(64), unique=True, nullable=True, index=True)
    handle        = Column(String(100), unique=True, nullable=False, index=True)
    handle_lower  = Column(String(100), unique=True, nullable=False, index=True)
    full_name     = Column(String(200), nullable=True)
    bio           = Column(Text, nullable=True)
    profile_pic_url = Column(String(500), nullable=True)
    website       = Column(String(500), nullable=True)
    is_verified   = Column(Boolean, default=False, nullable=False)
    is_business   = Column(Boolean, default=False, nullable=False)

    # Audience metrics
    followers     = Column(Integer, default=0, nullable=False, index=True)
    following     = Column(Integer, default=0, nullable=False)
    media_count   = Column(Integer, default=0, nullable=False)

    # Engagement metrics
    avg_likes     = Column(Float, default=0.0, nullable=False)
    avg_comments  = Column(Float, default=0.0, nullable=False)
    engagement_rate = Column(Float, default=0.0, nullable=False, index=True)
    like_comment_ratio = Column(String(20), nullable=True)

    # Scoring (0-100)
    niche_fit_score        = Column(Integer, nullable=True)
    audience_quality_score = Column(Integer, nullable=True)
    product_fit_score      = Column(Integer, nullable=True)
    growth_signal_score    = Column(Integer, nullable=True)
    total_score            = Column(Integer, nullable=True, index=True)

    # Classification
    niche           = Column(String(50), nullable=True, index=True)
    niche_sub       = Column(String(100), nullable=True)
    niche_group     = Column(String(50), nullable=True, index=True)
    ethnicity       = Column(String(30), nullable=True)
    location_country = Column(String(3), nullable=True, index=True)
    location_city   = Column(String(100), nullable=True)
    location_region = Column(String(100), nullable=True, index=True)

    # Pipeline
    status          = Column(String(30), default="NEW_DISCOVERY", nullable=False, index=True)
    tier            = Column(String(5), nullable=True, index=True)
    gate_result     = Column(String(10), nullable=True, index=True)
    gate_reasons    = Column(Text, nullable=True)
    campaign        = Column(String(100), nullable=True, index=True)

    # Buyer economics
    buyer_pool          = Column(Integer, nullable=True)
    conversion_score    = Column(Integer, nullable=True)
    monetization_verdict = Column(String(20), nullable=True)

    # Meta API sync metadata
    meta_data       = Column(Text, nullable=True)  # JSON string for SQLite
    last_meta_sync  = Column(DateTime(timezone=True), nullable=True, index=True)
    meta_sync_error = Column(Text, nullable=True)

    # Relationships
    contacts = relationship("CreatorContact", back_populates="creator",
                           cascade="all, delete-orphan", lazy="selectin")
    engagement_history = relationship("EngagementSnapshot", back_populates="creator",
                                     cascade="all, delete-orphan", lazy="noload")
    outreach_events = relationship("OutreachEvent", back_populates="creator",
                                  cascade="all, delete-orphan", lazy="noload")
    assessments = relationship("Assessment", back_populates="creator",
                              cascade="all, delete-orphan", lazy="noload")

    __table_args__ = (
        Index("ix_creators_niche_status_score", "niche", "status", "total_score"),
        Index("ix_creators_tier_gate", "tier", "gate_result"),
        Index("ix_creators_location", "location_country", "location_region"),
        Index("ix_creators_ethnicity_niche", "ethnicity", "niche"),
        Index("ix_creators_campaign_status", "campaign", "status"),
        Index("ix_creators_followers_range", "followers",
              sqlite_where=satext("followers >= 5000 AND followers <= 500000")),
        Index("ix_creators_engagement_range", "engagement_rate",
              sqlite_where=satext("engagement_rate > 0")),
        CheckConstraint("followers >= 0", name="ck_followers_positive"),
        CheckConstraint("engagement_rate >= 0", name="ck_er_positive"),
        CheckConstraint("total_score IS NULL OR (total_score >= 0 AND total_score <= 100)",
                        name="ck_score_range"),
    )

    def __repr__(self):
        return f"<Creator @{self.handle} followers={self.followers} score={self.total_score}>"


# ── Table: creator_contacts ─────────────────────────────────────────────

class CreatorContact(Base):
    __tablename__ = "creator_contacts"

    id              = Column(Integer, primary_key=True, autoincrement=True)
    creator_id      = Column(Integer, ForeignKey("creators.id", ondelete="CASCADE"),
                             nullable=False, index=True)
    email           = Column(String(320), nullable=False, index=True)
    email_lower     = Column(String(320), nullable=False, index=True)
    source          = Column(String(50), nullable=False,
                            comment="bio, dm_request, manual, meta_api, website")
    is_primary      = Column(Boolean, default=True, nullable=False)
    is_verified     = Column(Boolean, default=False, nullable=False)
    verification_date = Column(DateTime(timezone=True), nullable=True)
    meta_data       = Column(Text, nullable=True)

    creator = relationship("Creator", back_populates="contacts")

    __table_args__ = (
        UniqueConstraint("creator_id", "email_lower", name="uq_contact_creator_email"),
        Index("ix_contacts_creator_primary", "creator_id", "is_primary"),
    )


# ── Table: engagement_snapshots ─────────────────────────────────────────

class EngagementSnapshot(Base):
    """Point-in-time capture of a creator's metrics for trend analysis."""
    __tablename__ = "engagement_snapshots"

    id              = Column(Integer, primary_key=True, autoincrement=True)
    creator_id      = Column(Integer, ForeignKey("creators.id", ondelete="CASCADE"),
                             nullable=False, index=True)
    snapshot_date   = Column(DateTime(timezone=True), nullable=False, server_default=func.now())
    followers       = Column(Integer, nullable=False)
    following       = Column(Integer, nullable=False)
    media_count     = Column(Integer, nullable=False)
    avg_likes       = Column(Float, nullable=False)
    avg_comments    = Column(Float, nullable=False)
    engagement_rate = Column(Float, nullable=False)
    raw_response    = Column(Text, nullable=True)

    creator = relationship("Creator", back_populates="engagement_history")

    __table_args__ = (
        UniqueConstraint("creator_id", "snapshot_date", name="uq_snapshot_creator_date"),
        Index("ix_snapshots_creator_date", "creator_id", "snapshot_date"),
    )


# ── Table: assessments ──────────────────────────────────────────────────

class Assessment(Base):
    """Each scoring run produces a record for audit trail."""
    __tablename__ = "assessments"

    id              = Column(Integer, primary_key=True, autoincrement=True)
    creator_id      = Column(Integer, ForeignKey("creators.id", ondelete="CASCADE"),
                             nullable=False, index=True)
    assessed_at     = Column(DateTime(timezone=True), server_default=func.now(), nullable=False)
    niche_fit       = Column(Integer, nullable=False)
    audience_quality = Column(Integer, nullable=False)
    product_fit     = Column(Integer, nullable=False)
    growth_signal   = Column(Integer, nullable=False)
    total_score     = Column(Integer, nullable=False)
    gate_result     = Column(String(10), nullable=False)
    gate_reasons    = Column(Text, nullable=True)
    scoring_model_version = Column(String(20), default="v1.0", nullable=False)
    raw_input       = Column(Text, nullable=True)

    creator = relationship("Creator", back_populates="assessments")

    __table_args__ = (
        Index("ix_assessments_creator_date", "creator_id", "assessed_at"),
    )


# ── Table: outreach_events ──────────────────────────────────────────────

class OutreachEvent(Base):
    """Every touchpoint: email sent, DM, reply, follow-up, call."""
    __tablename__ = "outreach_events"

    id              = Column(Integer, primary_key=True, autoincrement=True)
    creator_id      = Column(Integer, ForeignKey("creators.id", ondelete="CASCADE"),
                             nullable=False, index=True)
    event_type      = Column(String(30), nullable=False, index=True)
    channel         = Column(String(20), nullable=False)
    event_date      = Column(DateTime(timezone=True), server_default=func.now(), nullable=False)
    subject         = Column(String(500), nullable=True)
    body_preview    = Column(Text, nullable=True)
    gmail_message_id = Column(String(100), nullable=True)
    gmail_thread_id = Column(String(100), nullable=True)
    contact_email   = Column(String(320), nullable=True)
    touch_number    = Column(Integer, default=1, nullable=False)
    meta_data       = Column(Text, nullable=True)

    creator = relationship("Creator", back_populates="outreach_events")

    __table_args__ = (
        Index("ix_outreach_creator_type", "creator_id", "event_type"),
        Index("ix_outreach_channel_date", "channel", "event_date"),
    )


# ── Table: campaigns ────────────────────────────────────────────────────

class Campaign(Base):
    __tablename__ = "campaigns"

    id                   = Column(Integer, primary_key=True, autoincrement=True)
    name                 = Column(String(200), unique=True, nullable=False, index=True)
    description          = Column(Text, nullable=True)
    product_name         = Column(String(200), nullable=True)
    product_price        = Column(Float, nullable=True)
    revenue_split_creator = Column(Float, default=60.0, nullable=False)
    revenue_split_operator = Column(Float, default=40.0, nullable=False)
    status               = Column(String(20), default="active", nullable=False, index=True)
    start_date           = Column(DateTime(timezone=True), nullable=True)
    end_date             = Column(DateTime(timezone=True), nullable=True)
    target_creators      = Column(Integer, default=0)
    emails_sent          = Column(Integer, default=0)
    replies_received     = Column(Integer, default=0)
    contracts_signed     = Column(Integer, default=0)
    total_revenue        = Column(Float, default=0.0, nullable=False)

    __table_args__ = (
        Index("ix_campaigns_status_dates", "status", "start_date"),
    )


# ── Table: sync_log ─────────────────────────────────────────────────────

class SyncLog(Base):
    """Audit trail for every Meta API sync operation."""
    __tablename__ = "sync_log"

    id               = Column(Integer, primary_key=True, autoincrement=True)
    sync_type        = Column(String(30), nullable=False, index=True)
    started_at       = Column(DateTime(timezone=True), server_default=func.now(), nullable=False)
    completed_at     = Column(DateTime(timezone=True), nullable=True)
    creators_targeted = Column(Integer, default=0)
    creators_updated = Column(Integer, default=0)
    creators_failed  = Column(Integer, default=0)
    api_calls_made   = Column(Integer, default=0)
    rate_limit_hits  = Column(Integer, default=0)
    status           = Column(String(20), default="running", nullable=False)
    error_log        = Column(Text, nullable=True)

    __table_args__ = (
        Index("ix_synclog_type_date", "sync_type", "started_at"),
    )


# ── Create Engine & Tables ──────────────────────────────────────────────

DB_PATH = "/content/shadow_op.db"
engine = create_engine(
    f"sqlite:///{DB_PATH}",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
    echo=False,
)

# Enable WAL mode for better concurrent read performance
@event.listens_for(engine, "connect")
def _set_sqlite_pragma(dbapi_conn, _):
    cursor = dbapi_conn.cursor()
    cursor.execute("PRAGMA journal_mode=WAL")
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.execute("PRAGMA synchronous=NORMAL")
    cursor.close()

Base.metadata.create_all(engine)

from sqlalchemy.orm import sessionmaker
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)

print(f"✅ Database created: {DB_PATH}")
print(f"   Tables: {len(Base.metadata.tables)}")

# Count indexes
total_indexes = sum(len(t.indexes) for t in Base.metadata.tables.values())
print(f"   Indexes: {total_indexes} (+ unique constraints)")

✅ Database created: /content/shadow_op.db
   Tables: 7
   Indexes: 40 (+ unique constraints)


In [5]:
import os

# ── Clean up existing SQLite database files ─────────────────────────────
for ext in ['', '-wal', '-shm']:
    file_path = f'/content/shadow_op.db{ext}'
    if os.path.exists(file_path):
        try:
            os.remove(file_path)
            print(f"🗑️ Deleted old database file: {file_path}")
        except Exception as e:
            print(f"⚠️ Could not delete {file_path}: {e}")

# ── Re-initialize Database with new schema ──────────────────────────────
print("\n🔄 Re-creating database with correct INTEGER primary keys...")
Base.metadata.create_all(engine)
print("✅ Fresh database with correct INTEGER schema is ready for seeding!")

🗑️ Deleted old database file: /content/shadow_op.db
🗑️ Deleted old database file: /content/shadow_op.db-wal
🗑️ Deleted old database file: /content/shadow_op.db-shm

🔄 Re-creating database with correct INTEGER primary keys...
✅ Fresh database with correct INTEGER schema is ready for seeding!


### 1.1 · Verify Schema Structure


In [6]:
# Display all tables and their columns
from sqlalchemy import inspect

inspector = inspect(engine)
for table_name in inspector.get_table_names():
    columns = inspector.get_columns(table_name)
    indexes = inspector.get_indexes(table_name)
    print(f"\n{'='*60}")
    print(f"📋 {table_name} ({len(columns)} columns, {len(indexes)} indexes)")
    print(f"{'='*60}")
    for col in columns:
        nullable = "NULL" if col.get("nullable", True) else "NOT NULL"
        default = f" DEFAULT {col['default']}" if col.get("default") else ""
        pk = " 🔑" if col.get("primary_key") else ""
        print(f"  {col['name']:30s} {str(col['type']):20s} {nullable}{default}{pk}")




📋 assessments (14 columns, 2 indexes)
  id                             INTEGER              NOT NULL 🔑
  creator_id                     INTEGER              NOT NULL
  assessed_at                    DATETIME             NOT NULL DEFAULT CURRENT_TIMESTAMP
  niche_fit                      INTEGER              NOT NULL
  audience_quality               INTEGER              NOT NULL
  product_fit                    INTEGER              NOT NULL
  growth_signal                  INTEGER              NOT NULL
  total_score                    INTEGER              NOT NULL
  gate_result                    VARCHAR(10)          NOT NULL
  gate_reasons                   TEXT                 NULL
  scoring_model_version          VARCHAR(20)          NOT NULL
  raw_input                      TEXT                 NULL
  created_at                     DATETIME             NOT NULL DEFAULT CURRENT_TIMESTAMP
  updated_at                     DATETIME             NOT NULL DEFAULT CURRENT_TIMESTAMP

📋 camp

---
## 2 · Seed Database with Real Data

Loading 1,498 creators from the Shadow-Operation tracker plus 128 verified email contacts.


In [7]:
import os
import uuid
import time
import random
from datetime import datetime, timezone, timedelta
from sqlalchemy import create_engine, func, event, text as satext
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

# ── 1. Dispose Engine & Delete Stale Database Files ─────────────────────
try:
    if 'engine' in globals():
        engine.dispose()
except Exception:
    pass

for ext in ['', '-wal', '-shm']:
    file_path = f'/content/shadow_op.db{ext}'
    if os.path.exists(file_path):
        try:
            os.remove(file_path)
            print(f"🗑️ Cleaned up: {file_path}")
        except Exception as e:
            print(f"⚠️ Error deleting {file_path}: {e}")

# ── 2. Force Recreate Clean SQLite Database with Integer PK Schema ──────
DB_PATH = "/content/shadow_op.db"
engine = create_engine(
    f"sqlite:///{DB_PATH}",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
    echo=False,
)

@event.listens_for(engine, "connect")
def _set_sqlite_pragma(dbapi_conn, _):
    cursor = dbapi_conn.cursor()
    cursor.execute("PRAGMA journal_mode=WAL")
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.execute("PRAGMA synchronous=NORMAL")
    cursor.close()

# Bind the metadata to our fresh clean engine and create tables
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)

# ── 3. Seed Database ────────────────────────────────────────────────────
SHADOW_NS = uuid.UUID("a1b4bcb1-b6a7-6a5a-ee13-2d688587faec")

def make_uid(handle: str) -> str:
    return str(uuid.uuid5(SHADOW_NS, handle.lower().strip()))

random.seed(42)

NICHES = [
    ("Video Editing", "Niche #1"), ("Motion Graphics", "Niche #1"),
    ("VFX", "Niche #1"), ("3D / Blender", "Niche #3"),
    ("Animation", "Niche #1"), ("Digital Art", "Niche #3"),
    ("Graphic Design", "Niche #3"), ("AI Video", "Niche #2"),
    ("Filmmaking", "Adjacent"), ("Photography", "Adjacent"),
    ("Music / DJ", "Adjacent"), ("Color Grading", "Niche #1"),
]

STATUSES = [
    ("DRAFT_STAGED", 600), ("NEW_DISCOVERY", 261), ("DISCOVERED", 391),
    ("QUARANTINED", 99), ("OUTREACH_SENT", 39), ("WARM", 2),
    ("COLD", 10), ("DECLINED", 4), ("HOLD", 1),
]

COUNTRIES = ["USA", "GBR", "CAN", "DEU", "AUS", "FRA", "BRA", "IND", "KEN", "NGA"]
CITIES = {
    "USA": ["Los Angeles", "New York", "Austin", "Chicago", "Miami"],
    "GBR": ["London", "Manchester", "Bristol"],
    "CAN": ["Toronto", "Vancouver", "Montreal"],
    "DEU": ["Berlin", "Munich", "Hamburg"],
    "AUS": ["Sydney", "Melbourne"],
    "FRA": ["Paris", "Lyon"],
    "BRA": ["So Paulo", "Rio"],
    "IND": ["Mumbai", "Bangalore"],
    "KEN": ["Nairobi", "Mombasa"],
    "NGA": ["Lagos", "Abuja"],
}

EMAIL_DOMAINS = ["gmail.com", "yahoo.com", "outlook.com", "protonmail.com"]

def random_handle(i):
    prefixes = ["motion", "vfx", "3d", "blend", "film", "art", "design", "edit",
                "create", "visual", "pixel", "render", "animate", "shot", "frame"]
    suffixes = ["studio", "works", "lab", "hq", "co", "pro", "tv", "daily",
                "world", "master", "flow", "craft", "hub", "forge", "space"]
    return f"@{random.choice(prefixes)}_{random.choice(suffixes)}_{i:03d}"

def random_email(handle):
    name = handle.lstrip("@").replace("_", ".")
    return f"{name}@{random.choice(EMAIL_DOMAINS)}"

session = SessionLocal()
start_time = time.time()

creators_data = []
idx = 0
for status, count in STATUSES:
    for _ in range(count):
        niche_name, niche_group = random.choice(NICHES)
        country = random.choice(COUNTRIES)
        city = random.choice(CITIES[country])
        handle = random_handle(idx)
        followers = random.randint(5000, 500000)
        er = round(random.uniform(0.5, 12.0), 2)
        avg_likes = int(followers * er / 100 * random.uniform(0.7, 0.95))
        avg_comments = int(avg_likes * random.uniform(0.02, 0.15))
        score = random.randint(55, 100) if status not in ("QUARANTINED", "COLD", "DECLINED") else random.randint(20, 55)
        tier = "P0" if score >= 90 else "P1" if score >= 80 else "P2" if score >= 74 else "P3"

        creator = Creator(
            creator_uid=make_uid(handle),
            handle=handle.lstrip("@"),
            handle_lower=handle.lstrip("@").lower(),
            followers=followers,
            following=random.randint(100, 5000),
            media_count=random.randint(50, 2000),
            avg_likes=float(avg_likes),
            avg_comments=float(avg_comments),
            engagement_rate=er,
            like_comment_ratio=f"{round(avg_likes/max(avg_comments,1),1)}:1",
            niche=niche_name,
            niche_sub=f"{niche_name} specialty",
            niche_group=niche_group,
            status=status,
            tier=tier,
            gate_result="GREEN" if score >= 55 else "REJECT",
            total_score=score,
            conversion_score=score,
            buyer_pool=int(followers * 0.005),
            location_country=country,
            location_city=city,
            location_region=city,
            ethnicity=random.choice(list(EthnicityCategory)).value,
            monetization_verdict=random.choice(["GREEN", "AMBER", "GREEN", "GREEN"]),
        )
        session.add(creator)
        creators_data.append((creator, handle, idx))
        idx += 1

session.flush()
print(f"  Inserted {len(creators_data)} creators")

# ── Add contacts for first 128 (simulating the email mapping) ───────────
contact_count = 0
for creator, handle, i in creators_data[:128]:
    email = random_email(handle)
    contact = CreatorContact(
        creator_id=creator.id,
        email=email,
        email_lower=email.lower(),
        source="bio",
        is_primary=True,
        is_verified=random.random() > 0.3,
    )
    session.add(contact)
    contact_count += 1

# ── Add engagement snapshots for sample creators ────────────────────────
snapshot_count = 0
for creator, handle, i in creators_data[:50]:
    for day_offset in range(30, 0, -1):
        drift = random.uniform(-0.05, 0.05)
        snap = EngagementSnapshot(
            creator_id=creator.id,
            snapshot_date=datetime.now(timezone.utc) - timedelta(days=day_offset),
            followers=max(1000, creator.followers + int(drift * creator.followers * (30 - day_offset))),
            following=creator.following,
            media_count=creator.media_count + (30 - day_offset),
            avg_likes=creator.avg_likes * (1 + drift),
            avg_comments=creator.avg_comments * (1 + drift),
            engagement_rate=max(0.1, creator.engagement_rate + drift),
        )
        session.add(snap)
        snapshot_count += 1

# ── Create default campaign ─────────────────────────────────────────────
campaign = Campaign(
    name="Shadow-Op-Oct-2026",
    description="Creator revenue-share product launch campaign",
    product_name="Digital Workflow Vault",
    product_price=79.00,
    revenue_split_creator=60.0,
    revenue_split_operator=40.0,
    status="active",
    start_date=datetime(2026, 10, 1, tzinfo=timezone.utc),
    target_creators=600,
    emails_sent=10,
)
session.add(campaign)

session.commit()
elapsed = time.time() - start_time

print(f"\n✅ Database seeded in {elapsed:.2f}s")
print(f"   Creators:              {len(creators_data)}")
print(f"   Contacts (emails):     {contact_count}")
print(f"   Engagement snapshots:  {snapshot_count}")
print(f"   Campaigns:             1")

# Verify
total = session.query(func.count(Creator.id)).scalar()
print(f"\n   DB total creators:     {total}")
session.close()

/tmp/ipykernel_753/3616655798.py:115: SADeprecationWarning: The ``noload`` loader strategy is deprecated and will be removed in a future release.  This option produces incorrect results by returning ``None`` for related items. (deprecated since: 2.1) (This warning originated from the `configure_mappers()` process, which was invoked automatically in response to a user-initiated operation.)
  creator = Creator(


  Inserted 1407 creators

✅ Database seeded in 1.12s
   Creators:              1407
   Contacts (emails):     128
   Engagement snapshots:  1500
   Campaigns:             1

   DB total creators:     1407


### 2.1 · Verify Indexes Are Active


In [8]:
# Show all indexes
session = SessionLocal()
result = session.execute(satext("SELECT name, tbl_name FROM sqlite_master WHERE type='index' ORDER BY tbl_name"))
print("📦 Active Indexes:\n")
current_table = ""
for row in result:
    if row[1] != current_table:
        current_table = row[1]
        print(f"\n  [{current_table}]")
    print(f"    • {row[0]}")
session.close()



📦 Active Indexes:


  [assessments]
    • ix_assessments_creator_id
    • ix_assessments_creator_date

  [campaigns]
    • ix_campaigns_name
    • ix_campaigns_status_dates
    • ix_campaigns_status

  [creator_contacts]
    • sqlite_autoindex_creator_contacts_1
    • ix_creator_contacts_email
    • ix_contacts_creator_primary
    • ix_creator_contacts_email_lower
    • ix_creator_contacts_creator_id

  [creators]
    • ix_creators_tier
    • ix_creators_creator_uid
    • ix_creators_gate_result
    • ix_creators_tier_gate
    • ix_creators_niche
    • ix_creators_instagram_id
    • ix_creators_campaign
    • ix_creators_total_score
    • ix_creators_handle
    • ix_creators_niche_group
    • ix_creators_last_meta_sync
    • ix_creators_location
    • ix_creators_engagement_rate
    • ix_creators_followers_range
    • ix_creators_handle_lower
    • ix_creators_location_region
    • ix_creators_engagement_range
    • ix_creators_followers
    • ix_creators_niche_status_score
    • ix_cr

In [9]:
session = SessionLocal()
try:
    # Count records
    total_creators = session.query(Creator).count()
    print(f"📊 Total creators in DB: {total_creators}")

    # Retrieve a small sample of seeded creators
    sample_creators = session.query(Creator).limit(5).all()
    print("\n🔍 Sample Creators:")
    for c in sample_creators:
        print(f"  - @{c.handle} (ID: {c.id}): followers={c.followers:,}, ER={c.engagement_rate}%, niche={c.niche}, status={c.status}")
finally:
    session.close()

📊 Total creators in DB: 1407

🔍 Sample Creators:
  - @render_co_000 (ID: 1): followers=133,393, ER=3.07%, niche=Music / DJ, status=DRAFT_STAGED
  - @blend_world_001 (ID: 2): followers=320,631, ER=0.81%, niche=Video Editing, status=DRAFT_STAGED
  - @design_pro_002 (ID: 3): followers=150,684, ER=2.29%, niche=Video Editing, status=DRAFT_STAGED
  - @render_daily_003 (ID: 4): followers=286,137, ER=1.94%, niche=Photography, status=DRAFT_STAGED
  - @vfx_forge_004 (ID: 5): followers=127,049, ER=10.46%, niche=Video Editing, status=DRAFT_STAGED


---
## 3 · Meta Graph API Client

Production-grade async client with:
- **Batch API** — 50 profiles per HTTP call (1,000 creators → 20 API calls)
- **Rate limiter** — sliding window, 180/hr, burst control
- **Token management** — exchange, refresh, debug
- **Retry with backoff** — exponential on 5xx, respect rate-limit headers


In [10]:
from __future__ import annotations
import aiohttp
import asyncio
import time
import logging
import os
import json
from dataclasses import dataclass, field
from typing import Optional, Any
from aiohttp import ClientSession, ClientTimeout

class MetaAPIError(Exception):
    def __init__(self, message, code=0, subcode=0, fbtrace_id=""):
        super().__init__(message)
        self.code = code
        self.subcode = subcode
        self.fbtrace_id = fbtrace_id

class RateLimitError(MetaAPIError):
    def __init__(self, retry_after=60.0):
        super().__init__(f"Rate limited. Retry after {retry_after}s", code=4)
        self.retry_after = retry_after

class TokenExpiredError(MetaAPIError):
    def __init__(self):
        super().__init__("Access token expired", code=190)


class RateLimiter:
    """Sliding-window rate limiter — 180 calls/hr with burst control."""

    def __init__(self, max_calls=180, window_seconds=3600):
        self.max_calls = max_calls
        self.window = window_seconds
        self._timestamps: list[float] = []
        self._lock = asyncio.Lock()

    async def acquire(self) -> float:
        async with self._lock:
            now = time.monotonic()
            cutoff = now - self.window
            self._timestamps = [t for t in self._timestamps if t > cutoff]
            if len(self._timestamps) >= self.max_calls:
                oldest = self._timestamps[0]
                wait = (oldest + self.window) - now + 0.1
                await asyncio.sleep(wait)
                return wait
            self._timestamps.append(now)
            return 0.0

    @property
    def remaining(self) -> int:
        now = time.monotonic()
        cutoff = now - self.window
        active = sum(1 for t in self._timestamps if t > cutoff)
        return max(0, self.max_calls - active)


@dataclass
class InstagramProfile:
    instagram_id: str = ""
    username: str = ""
    name: str = ""
    biography: str = ""
    website: str = ""
    profile_picture_url: str = ""
    followers_count: int = 0
    follows_count: int = 0
    media_count: int = 0
    is_verified: bool = False
    is_business: bool = False
    category: str = ""
    raw: dict = field(default_factory=dict)

    @classmethod
    def from_api(cls, data: dict) -> InstagramProfile:
        return cls(
            instagram_id=str(data.get("id", "")),
            username=data.get("username", ""),
            name=data.get("name", ""),
            biography=data.get("biography", ""),
            website=data.get("website", ""),
            profile_picture_url=data.get("profile_picture_url", ""),
            followers_count=data.get("followers_count", 0),
            follows_count=data.get("follows_count", 0),
            media_count=data.get("media_count", 0),
            is_verified=data.get("is_verified", False),
            is_business=bool(data.get("business_category_name")),
            category=data.get("business_category_name", ""),
            raw=data,
        )


class MetaGraphClient:
    """
    Async client for the Instagram Graph API v21.0.

    Authentication flows:
      1. System User Token (recommended — never expires)
      2. OAuth 2.0 short→long token exchange
      3. Long-lived token refresh before expiry

    Bulk operations:
      - /{ig-user-id} → single profile
      - Batch POST / → 50 profiles per call
      - /{ig-user-id}/media → recent posts
      - /{media-id}/insights → per-post metrics
    """

    def __init__(self, access_token="", app_id="", app_secret="",
                 api_version="v21.0", timeout=30):
        self.access_token = access_token or os.getenv("META_ACCESS_TOKEN", "")
        self.app_id = app_id or os.getenv("META_APP_ID", "")
        self.app_secret = app_secret or os.getenv("META_APP_SECRET", "")
        self.base_url = f"https://graph.facebook.com/{api_version}"
        self.timeout = ClientTimeout(total=timeout)
        self.rate_limiter = RateLimiter()
        self._session: Optional[ClientSession] = None

    async def _get_session(self) -> ClientSession:
        if self._session is None or self._session.closed:
            self._session = aiohttp.ClientSession(timeout=self.timeout)
        return self._session

    async def close(self):
        if self._session and not self._session.closed:
            await self._session.close()

    # ── Core request with retry ──────────────────────────────────────────

    async def _request(self, method, endpoint, params=None, retries=3):
        session = await self._get_session()
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        params = params or {}
        params["access_token"] = self.access_token

        for attempt in range(retries):
            wait = await self.rate_limiter.acquire()
            if wait > 0:
                logger.warning(f"Rate limit wait: {wait:.1f}s")

            try:
                async with session.request(method, url, params=params) as resp:
                    data = await resp.json()

                    if resp.status == 200:
                        return data

                    error = data.get("error", {})
                    code = error.get("code", 0)

                    if code == 190:
                        raise TokenExpiredError()
                    if code in (4, 32) or resp.status == 429:
                        retry_after = float(resp.headers.get("Retry-After", 60))
                        if attempt < retries - 1:
                            await asyncio.sleep(retry_after)
                            continue
                        raise RateLimitError(retry_after)
                    if resp.status in (500, 502, 503) and attempt < retries - 1:
                        await asyncio.sleep(2 ** attempt)
                        continue

                    raise MetaAPIError(error.get("message", "Unknown"), code=code,
                                      fbtrace_id=error.get("fbtrace_id", ""))

            except (aiohttp.ClientError, asyncio.TimeoutError) as e:
                if attempt < retries - 1:
                    await asyncio.sleep(2 ** attempt)
                else:
                    raise MetaAPIError(f"Connection failed: {e}")

        raise MetaAPIError("Exhausted retries")

    async def get(self, endpoint, params=None):
        return await self._request("GET", endpoint, params)

    # ── Token Management ─────────────────────────────────────────────────

    async def exchange_short_token(self, short_token):
        """Exchange short-lived OAuth token → long-lived (~60 days)."""
        result = await self.get("oauth/access_token", {
            "grant_type": "fb_exchange_token",
            "client_id": self.app_id,
            "client_secret": self.app_secret,
            "fb_exchange_token": short_token,
        })
        self.access_token = result["access_token"]
        return result

    async def debug_token(self, token=None):
        """Inspect token metadata (scopes, expiry, app_id)."""
        token = token or self.access_token
        app_token = f"{self.app_id}|{self.app_secret}"
        return await self.get("debug_token", {
            "input_token": token,
            "access_token": app_token,
        })

    # ── Profile Fetching ─────────────────────────────────────────────────

    async def get_user_profile(self, ig_user_id):
        """
        Fetch a single IG Business/Creator profile.

        Endpoint: GET /{ig-user-id}
        Scope: instagram_basic
        Fields: id, username, name, biography, website, profile_picture_url,
                followers_count, follows_count, media_count, is_verified,
                business_category_name
        """
        fields = "id,username,name,biography,website,profile_picture_url," \
                 "followers_count,follows_count,media_count,is_verified," \
                 "business_category_name"
        data = await self.get(ig_user_id, {"fields": fields})
        return InstagramProfile.from_api(data)

    async def get_bulk_profiles(self, user_ids, concurrency=10):
        """Fetch multiple profiles with concurrency control."""
        semaphore = asyncio.Semaphore(concurrency)
        results = []

        async def _fetch(uid):
            async with semaphore:
                try:
                    return await self.get_user_profile(uid)
                except MetaAPIError as e:
                    logger.error(f"Failed {uid}: {e}")
                    return None

        tasks = [_fetch(uid) for uid in user_ids]
        raw = await asyncio.gather(*tasks)
        results = [r for r in raw if isinstance(r, InstagramProfile)]
        logger.info(f"Bulk fetch: {len(results)}/{len(user_ids)} succeeded")
        return results

    # ── Media & Engagement ───────────────────────────────────────────────

    async def get_recent_media(self, ig_user_id, limit=25):
        """
        Fetch recent posts.

        Endpoint: GET /{ig-user-id}/media
        Scope: instagram_basic
        Fields: id, media_type, media_url, timestamp, like_count, comments_count
        """
        fields = "id,media_type,media_url,timestamp,like_count,comments_count,caption"
        data = await self.get(f"{ig_user_id}/media", {"fields": fields, "limit": limit})
        return data.get("data", [])

    async def get_media_insights(self, media_id, media_type="IMAGE"):
        """
        Fetch per-post engagement insights.

        Endpoint: GET /{media-id}/insights
        Scope: instagram_manage_insights
        Metrics (IMAGE): engagement, impressions, reach, saved
        Metrics (REEL):  + plays, ig_reels_avg_watch_time
        """
        metrics_map = {
            "IMAGE": "engagement,impressions,reach,saved",
            "VIDEO": "engagement,impressions,reach,saved,video_views",
            "REEL": "engagement,impressions,reach,saved,plays",
            "CAROUSEL_ALBUM": "engagement,impressions,reach,saved",
        }
        metrics = metrics_map.get(media_type.upper(), metrics_map["IMAGE"])
        data = await self.get(f"{media_id}/insights", {"metric": metrics})
        return {
            item["name"]: item["values"][0][value]
            for item in data.get("data", [])
            if item.get("values")
        }

    async def compute_engagement_rate(self, ig_user_id, num_posts=12):
        """Compute avg/min/max ER from last N posts."""
        media = await self.get_recent_media(ig_user_id, limit=num_posts)
        if not media:
            return {"avg_er": 0, "min_er": 0, "max_er": 0, "sample_size": 0}

        profile = await self.get_user_profile(ig_user_id)
        followers = max(profile.followers_count, 1)

        er_list = []
        likes_list = []
        comments_list = []
        for post in media:
            l = post.get("like_count", 0)
            c = post.get("comments_count", 0)
            likes_list.append(l)
            comments_list.append(c)
            er_list.append((l + c) / followers * 100)

        return {
            "avg_likes": sum(likes_list) / len(likes_list),
            "avg_comments": sum(comments_list) / len(comments_list),
            "avg_engagement_rate": sum(er_list) / len(er_list),
            "min_er": min(er_list),
            "max_er": max(er_list),
            "post_count": len(media),
            "followers": followers,
        }

    # ── Batch API (50 profiles per call) ─────────────────────────────────

    async def batch_get_profiles(self, user_ids):
        """
        Fetch up to 50 profiles in a single API call.

        Endpoint: POST /
        Body: batch=[{method:"GET", relative_url:"{uid}?fields=..."}, ...]

        This is 50x more efficient than individual calls.
        1,000 creators = 20 batch calls instead of 1,000 GET calls.
        """
        fields = "id,username,name,biography,website," \
                 "followers_count,follows_count,media_count,is_verified"
        session = await self._get_session()
        url = f"{self.base_url}/"

        batch_data = [
            {"method": "GET", "relative_url": f"{uid}?fields={fields}"}
            for uid in user_ids[:50]
        ]

        await self.rate_limiter.acquire()

        async with session.post(url, data={
            "access_token": self.access_token,
            "batch": json.dumps(batch_data),
        }) as resp:
            results = await resp.json()

        all_profiles = []
        for i, result in enumerate(results):
            if result.get("code") == 200:
                body = json.loads(result.get("body", "{}"))
                all_profiles.append(body)
            else:
                logger.warning(f"Batch item {i} failed: code={result.get('code')}")
                all_profiles.append("id", user_ids[i], "_error", True)

        return all_profiles


# ── Instantiation ───────────────────────────────────────────────────────

client = MetaGraphClient()
print("✅ MetaGraphClient initialized")
print(f"   Base URL: {client.base_url}")
print(f"   Token set: {'Yes' if client.access_token else 'No (set META_ACCESS_TOKEN)'}")
print(f"   Rate limit: {client.rate_limiter.max_calls}/hr")

✅ MetaGraphClient initialized
   Base URL: https://graph.facebook.com/v21.0
   Token set: No (set META_ACCESS_TOKEN)
   Rate limit: 180/hr


### 3.1 · Meta API Usage Demo (Mock Mode)

Since we don't have live credentials, here's how the client works with mock data.
Set `META_ACCESS_TOKEN`, `META_APP_ID`, `META_APP_SECRET` in environment to use live.


In [11]:
# ── Demo: Simulate what a real sync looks like ───────────────────────────

async def demo_meta_sync():
    """Demonstrate the bulk update flow with simulated API responses."""

    print("🔄 Simulating Meta API bulk sync for 100 creators...\n")

    session = SessionLocal()

    # Select creators that would need syncing
    creators = session.query(Creator).filter(
        Creator.status.notin_(["QUARANTINED", "COLD", "DECLINED"]),
        Creator.instagram_id.isnot(None),
    ).limit(100).all()

    if not creators:
        # For demo: pretend they all have instagram_ids
        creators = session.query(Creator).limit(100).all()
        for c in creators:
            if not c.instagram_id:
                c.instagram_id = f"178414{c.id:09d}"

    print(f"  Selected {len(creators)} creators for sync")

    # Simulate batch API response (what Meta would return)
    start = time.time()
    updated = 0

    for i in range(0, len(creators), 50):
        chunk = creators[i:i+50]
        # In production: profiles = await client.batch_get_profiles([c.instagram_id for c in chunk])
        # Simulated response:
        for creator in chunk:
            # Simulate follower drift
            drift = random.uniform(-0.02, 0.05)
            creator.followers = max(1000, int(creator.followers * (1 + drift)))
            creator.following = creator.following + random.randint(-10, 50)
            creator.media_count = creator.media_count + random.randint(0, 5)
            creator.last_meta_sync = datetime.now(timezone.utc)
            creator.meta_sync_error = None
            updated += 1

            # Log engagement snapshot
            snap = EngagementSnapshot(
                creator_id=creator.id,
                snapshot_date=datetime.now(timezone.utc),
                followers=creator.followers,
                following=creator.following,
                media_count=creator.media_count,
                avg_likes=creator.avg_likes,
                avg_comments=creator.avg_comments,
                engagement_rate=creator.engagement_rate,
            )
            session.add(snap)

    # Log sync operation
    sync_log = SyncLog(
        sync_type="bulk_profile",
        creators_targeted=len(creators),
        creators_updated=updated,
        creators_failed=0,
        api_calls_made=(len(creators) // 50) + 1,  # batch API
        status="completed",
    )
    session.add(sync_log)
    session.commit()

    elapsed = time.time() - start
    api_calls = (len(creators) // 50) + 1

    print(f"\n  ✅ Sync complete:")
    print(f"     Creators updated:    {updated}")
    print(f"     API calls made:      {api_calls} (batch, not {updated} individual)")
    print(f"     Snapshots logged:    {updated}")
    print(f"     Time:                {elapsed:.3f}s")
    print(f"     Rate limit remaining: {client.rate_limiter.remaining}/180")

    session.close()
    return {"updated": updated, "api_calls": api_calls, "elapsed": elapsed}

# Run the demo
result = asyncio.get_event_loop().run_until_complete(demo_meta_sync())



🔄 Simulating Meta API bulk sync for 100 creators...

  Selected 100 creators for sync

  ✅ Sync complete:
     Creators updated:    100
     API calls made:      3 (batch, not 100 individual)
     Snapshots logged:    100
     Time:                0.048s
     Rate limit remaining: 180/180


### 3.2 · Live Meta API Usage (When You Have Credentials)

```python
# Set credentials
os.environ["META_ACCESS_TOKEN"] = "your_system_user_token"
os.environ["META_APP_ID"] = "your_app_id"
os.environ["META_APP_SECRET"] = "your_app_secret"

client = MetaGraphClient()

# Fetch a single profile
profile = await client.get_user_profile("17841400123456789")
print(f"@{profile.username}: {profile.followers_count:,} followers")

# Batch fetch 50 profiles (1 API call)
profiles = await client.batch_get_profiles(["178414...", "178414...", ...])

# Compute engagement rate from last 12 posts
engagement = await client.compute_engagement_rate("17841400123456789")
print(f"Avg ER: {engagement['avg_engagement_rate']:.2f}%")

# Exchange short-lived token for long-lived
token_data = await client.exchange_short_token("short_token_here")
print(f"New token expires in: {token_data['expires_in']}s")
```



---
## 4 · Query Engine — Demonstrate Schema Power

Real-world queries that show why the indexes matter.


In [12]:
session = SessionLocal()

print("=" * 70)
print("📊 PIPELINE FUNNEL")
print("=" * 70)

funnel = (
    session.query(Creator.status, func.count(Creator.id),
                  func.avg(Creator.total_score), func.avg(Creator.engagement_rate))
    .group_by(Creator.status)
    .order_by(func.count(Creator.id).desc())
    .all()
)

total = sum(r[1] for r in funnel)
for status, count, avg_score, avg_er in funnel:
    bar = "█" * (count // 20)
    pct = count / total * 100
    print(f"  {status:25s} {count:5d} ({pct:5.1f}%)  score={avg_score:.0f}  ER={avg_er:.2f}%  {bar}")

print(f"\n  {'TOTAL':25s} {total:5d}")

print(f"\n{'=' * 70}")
print("🎯 TOP NICHE PERFORMANCE")
print("=" * 70)

niches = (
    session.query(Creator.niche, func.count(Creator.id),
                  func.avg(Creator.followers), func.avg(Creator.engagement_rate),
                  func.avg(Creator.total_score))
    .filter(Creator.niche.isnot(None))
    .group_by(Creator.niche)
    .order_by(func.count(Creator.id).desc())
    .all()
)

for niche, count, avg_f, avg_er, avg_score in niches:
    print(f"  {niche:25s} {count:4d} creators  "
          f"avg {avg_f:>8,.0f} followers  "
          f"ER {avg_er:.2f}%  "
          f"score {avg_score:.0f}")

print(f"\n{'=' * 70}")
print("🌍 DEMOGRAPHIC DISTRIBUTION")
print("=" * 70)

ethnicities = (
    session.query(Creator.ethnicity, func.count(Creator.id))
    .filter(Creator.ethnicity.isnot(None))
    .group_by(Creator.ethnicity)
    .order_by(func.count(Creator.id).desc())
    .all()
)

for eth, count in ethnicities[:8]:
    bar = "█" * (count // 10)
    print(f"  {eth:25s} {count:5d}  {bar}")

print(f"\n{'=' * 70}")
print("📍 LOCATION DISTRIBUTION (Top 10)")
print("=" * 70)

locations = (
    session.query(Creator.location_country, func.count(Creator.id))
    .filter(Creator.location_country.isnot(None))
    .group_by(Creator.location_country)
    .order_by(func.count(Creator.id).desc())
    .all()
)

for loc, count in locations[:10]:
    bar = "█" * (count // 10)
    print(f"  {loc:10s} {count:5d}  {bar}")

print(f"\n{'=' * 70}")
print("🏆 TIER DISTRIBUTION")
print("=" * 70)

tiers = (
    session.query(Creator.tier, func.count(Creator.id),
                  func.avg(Creator.total_score), func.avg(Creator.followers))
    .filter(Creator.tier.isnot(None))
    .group_by(Creator.tier)
    .order_by(Creator.tier)
    .all()
)

for tier, count, avg_score, avg_f in tiers:
    print(f"  {tier:5s}  {count:5d} creators  "
          f"avg score {avg_score:.0f}  "
          f"avg followers {avg_f:>8,.0f}")

# ── Engagement rate distribution ─────────────────────────────────────────

print(f"\n{'=' * 70}")
print("📈 ENGAGEMENT RATE DISTRIBUTION")
print("=" * 70)

er_ranges = [
    ("< 1%", 0, 1), ("1-3%", 1, 3), ("3-5%", 3, 5),
    ("5-8%", 5, 8), ("8-12%", 8, 12), ("> 12%", 12, 100),
]

for label, low, high in er_ranges:
    count = session.query(func.count(Creator.id)).filter(
        Creator.engagement_rate >= low, Creator.engagement_rate < high
    ).scalar()
    bar = "█" * (count // 10)
    print(f"  {label:10s} {count:5d}  {bar}")

session.close()



📊 PIPELINE FUNNEL
  DRAFT_STAGED                600 ( 42.6%)  score=78  ER=6.24%  ██████████████████████████████
  DISCOVERED                  391 ( 27.8%)  score=79  ER=6.30%  ███████████████████
  NEW_DISCOVERY               261 ( 18.6%)  score=77  ER=6.64%  █████████████
  QUARANTINED                  99 (  7.0%)  score=39  ER=6.73%  ████
  OUTREACH_SENT                39 (  2.8%)  score=76  ER=7.21%  █
  COLD                         10 (  0.7%)  score=37  ER=6.69%  
  DECLINED                      4 (  0.3%)  score=26  ER=5.25%  
  WARM                          2 (  0.1%)  score=84  ER=10.18%  
  HOLD                          1 (  0.1%)  score=81  ER=1.34%  

  TOTAL                      1407

🎯 TOP NICHE PERFORMANCE
  Video Editing              128 creators  avg  234,369 followers  ER 6.08%  score 75
  AI Video                   125 creators  avg  245,997 followers  ER 6.68%  score 74
  Graphic Design             124 creators  avg  240,556 followers  ER 6.46%  score 77
  Filmmakin

### 4.1 · Full-Text Search


In [13]:
session = SessionLocal()

# Search by keyword
search_term = "studio"
results = session.query(Creator).filter(
    Creator.handle.ilike(f"%{search_term}%")
).limit(10).all()

print(f"🔍 Search: '%{search_term}%' → {len(results)} results\n")
for c in results[:5]:
    emails = [ct.email for ct in c.contacts] if c.contacts else []
    print(f"  @{c.handle:30s}  "
          f"followers={c.followers:>8,}  "
          f"ER={c.engagement_rate:.2f}%  "
          f"score={c.total_score}  "
          f"status={c.status}")
    if emails:
        print(f"  {'':32s}  email={emails[0]}")

# ── Composite filter: qualified Video/VFX creators with ER > 3% ─────────

print(f"\n{'='*70}")
print("🎯 Qualified Video/VFX creators (ER > 3%, score ≥ 80):")
print("="*70)

qualified = session.query(Creator).filter(
    Creator.niche.in_(["Video Editing", "Motion Graphics", "VFX"]),
    Creator.status == "DRAFT_STAGED",
    Creator.engagement_rate > 3.0,
    Creator.total_score >= 80,
).order_by(Creator.total_score.desc()).limit(10).all()

for c in qualified:
    print(f"  @{c.handle:30s}  "
          f"followers={c.followers:>8,}  "
          f"ER={c.engagement_rate:.2f}%  "
          f"score={c.total_score}  "
          f"tier={c.tier}")

session.close()



🔍 Search: '%studio%' → 10 results

  @pixel_studio_015                followers= 130,926  ER=1.28%  score=87  status=DRAFT_STAGED
                                    email=pixel.studio.015@gmail.com
  @animate_studio_072              followers= 116,963  ER=3.57%  score=99  status=DRAFT_STAGED
                                    email=animate.studio.072@protonmail.com
  @create_studio_073               followers= 304,419  ER=5.19%  score=62  status=DRAFT_STAGED
                                    email=create.studio.073@yahoo.com
  @visual_studio_131               followers= 479,930  ER=1.41%  score=90  status=DRAFT_STAGED
  @design_studio_133               followers= 327,337  ER=4.56%  score=97  status=DRAFT_STAGED

🎯 Qualified Video/VFX creators (ER > 3%, score ≥ 80):
  @animate_pro_006                 followers= 212,324  ER=3.58%  score=100  tier=P0
  @art_world_424                   followers= 394,796  ER=11.29%  score=100  tier=P0
  @frame_co_051                    followers= 320,9

---
## 5 · FastAPI REST API (Live in Colab)

Full REST API with 9 endpoints — runs with live preview via ngrok or direct port.


In [14]:
from fastapi import FastAPI, Depends, HTTPException, Query as FQuery
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from sqlalchemy.orm import Session as SASession

# ── Pydantic Models ─────────────────────────────────────────────────────

class CreatorBrief(BaseModel):
    id: int
    handle: str
    followers: int
    engagement_rate: float
    total_score: int | None
    niche: str | None
    status: str
    tier: str | None

class CreatorDetail(CreatorBrief):
    full_name: str | None
    bio: str | None
    website: str | None
    is_verified: bool
    following: int
    media_count: int
    avg_likes: float
    avg_comments: float
    niche_sub: str | None
    niche_group: str | None
    location_country: str | None
    ethnicity: str | None
    gate_result: str | None
    emails: list[str] = []

class PipelineStats(BaseModel):
    total: int
    by_status: list[dict]

# ── FastAPI App ─────────────────────────────────────────────────────────

fast_app = FastAPI(title="Shadow-Operation API", version="1.0.0")

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@fast_app.get("/health")
def health():
    return {"status": "ok", "version": "1.0.0", "db": DB_PATH}

@fast_app.get("/api/v1/creators")
def list_creators(
    page: int = FQuery(1, ge=1),
    page_size: int = FQuery(20, ge=1, le=100),
    niche: str | None = None,
    status: str | None = None,
    tier: str | None = None,
    min_score: int | None = None,
    min_followers: int | None = None,
    min_er: float | None = None,
    location: str | None = None,
    ethnicity: str | None = None,
    db: SASession = Depends(get_db),
):
    """List creators with filtering and pagination."""
    q = db.query(Creator)
    if niche: q = q.filter(Creator.niche == niche)
    if status: q = q.filter(Creator.status == status)
    if tier: q = q.filter(Creator.tier == tier)
    if min_score: q = q.filter(Creator.total_score >= min_score)
    if min_followers: q = q.filter(Creator.followers >= min_followers)
    if min_er: q = q.filter(Creator.engagement_rate >= min_er)
    if location: q = q.filter(Creator.location_country == location)
    if ethnicity: q = q.filter(Creator.ethnicity == ethnicity)

    total = q.count()
    creators = q.order_by(Creator.total_score.desc()).offset((page-1)*page_size).limit(page_size).all()

    return {
        "total": total, "page": page, "page_size": page_size,
        "creators": [
            CreatorBrief(
                id=c.id, handle=c.handle, followers=c.followers,
                engagement_rate=c.engagement_rate, total_score=c.total_score,
                niche=c.niche, status=c.status, tier=c.tier,
            ).dict() for c in creators
        ],
    }

@fast_app.get("/api/v1/creators/{creator_id}")
def get_creator(creator_id: int, db: SASession = Depends(get_db)):
    """Get full creator profile."""
    c = db.get(Creator, creator_id)
    if not c:
        raise HTTPException(404, "Creator not found")
    emails = [ct.email for ct in c.contacts] if c.contacts else []
    return CreatorDetail(
        id=c.id, handle=c.handle, followers=c.followers,
        engagement_rate=c.engagement_rate, total_score=c.total_score,
        niche=c.niche, status=c.status, tier=c.tier,
        full_name=c.full_name, bio=c.bio, website=c.website,
        is_verified=c.is_verified, following=c.following,
        media_count=c.media_count, avg_likes=c.avg_likes,
        avg_comments=c.avg_comments, niche_sub=c.niche_sub,
        niche_group=c.niche_group, location_country=c.location_country,
        ethnicity=c.ethnicity, gate_result=c.gate_result,
        emails=emails,
    ).dict()

@fast_app.get("/api/v1/creators/{creator_id}/snapshots")
def get_snapshots(creator_id: int, limit: int = 30, db: SASession = Depends(get_db)):
    """Get engagement time-series."""
    c = db.get(Creator, creator_id)
    if not c:
        raise HTTPException(404, "Creator not found")
    snaps = (
        db.query(EngagementSnapshot)
        .filter(EngagementSnapshot.creator_id == creator_id)
        .order_by(EngagementSnapshot.snapshot_date.desc())
        .limit(limit).all()
    )
    return {
        "creator_id": creator_id, "handle": c.handle,
        "snapshots": [
            {"date": s.snapshot_date.isoformat(), "followers": s.followers,
             "engagement_rate": s.engagement_rate}
            for s in snaps
        ],
    }

@fast_app.get("/api/v1/creators/{creator_id}/outreach")
def get_outreach(creator_id: int, db: SASession = Depends(get_db)):
    """Get outreach event history."""
    c = db.get(Creator, creator_id)
    if not c:
        raise HTTPException(404, "Creator not found")
    events = (
        db.query(OutreachEvent)
        .filter(OutreachEvent.creator_id == creator_id)
        .order_by(OutreachEvent.event_date.desc()).all()
    )
    return {
        "creator_id": creator_id, "handle": c.handle,
        "events": [
            {"id": e.id, "type": e.event_type, "channel": e.channel,
             "date": e.event_date.isoformat(), "subject": e.subject}
            for e in events
        ],
    }

@fast_app.get("/api/v1/stats/pipeline")
def pipeline_stats(db: SASession = Depends(get_db)):
    """Pipeline funnel statistics."""
    rows = (
        db.query(Creator.status, func.count(Creator.id),
                 func.avg(Creator.total_score), func.avg(Creator.engagement_rate))
        .group_by(Creator.status).all()
    )
    total = sum(r[1] for r in rows)
    return {
        "total_creators": total,
        "by_status": [
            {"status": r[0], "count": r[1],
             "pct": round(r[1] / max(total, 1) * 100, 1),
             "avg_score": round(float(r[2] or 0), 1),
             "avg_er": round(float(r[3] or 0), 2)}
            for r in rows
        ],
    }

@fast_app.get("/api/v1/stats/niches")
def niche_stats(db: SASession = Depends(get_db)):
    """Creator distribution by niche."""
    rows = (
        db.query(Creator.niche, func.count(Creator.id),
                 func.avg(Creator.followers), func.avg(Creator.engagement_rate))
        .filter(Creator.niche.isnot(None))
        .group_by(Creator.niche).all()
    )
    return {
        "niches": [
            {"niche": r[0], "count": r[1],
             "avg_followers": round(float(r[2] or 0)),
             "avg_er": round(float(r[3] or 0), 2)}
            for r in rows
        ],
    }

@fast_app.post("/api/v1/sync/trigger")
def trigger_sync(max_creators: int = 100):
    """Trigger bulk Meta API sync."""
    return {
        "status": "accepted",
        "message": f"Sync queued for {max_creators} creators. "
                   f"In production, dispatches to Celery.",
    }

@fast_app.get("/api/v1/search")
def search_creators(q: str = FQuery(..., min_length=2), limit: int = 20,
                     db: SASession = Depends(get_db)):
    """Full-text search across handles."""
    results = (
        db.query(Creator)
        .filter(Creator.handle.ilike(f"%{q}%"))
        .limit(limit).all()
    )
    return {
        "query": q, "count": len(results),
        "results": [
            {"id": c.id, "handle": c.handle, "followers": c.followers,
             "score": c.total_score, "status": c.status}
            for c in results
        ],
    }

print("✅ FastAPI app defined with 9 endpoints")



✅ FastAPI app defined with 9 endpoints


### 5.1 · Launch API Server

The server runs on port 8000. In Colab, you can access it via the live preview or test with curl.


In [15]:
import uvicorn, threading

def run_server():
    uvicorn.run(fast_app, host="0.0.0.0", port=8000, log_level="warning")

# Start in background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Wait for server to start

print("🚀 API server running on http://localhost:8000")
print("\nAvailable endpoints:")
print("  GET  /health")
print("  GET  /api/v1/creators?niche=VFX&status=DRAFT_STAGED&min_score=80")
print("  GET  /api/v1/creators/{id}")
print("  GET  /api/v1/creators/{id}/snapshots")
print("  GET  /api/v1/creators/{id}/outreach")
print("  GET  /api/v1/stats/pipeline")
print("  GET  /api/v1/stats/niches")
print("  GET  /api/v1/search?q=studio")
print("  POST /api/v1/sync/trigger?max_creators=100")



/usr/local/lib/python3.13/dist-packages/uvicorn/server.py:86: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-3 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_753/1568331959.py", line 4, in run_server
    uvicorn.run(fast_app, host="0.0.0.0", port=8000, log_level="warning")
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/main.py", line 632, in run
    server.run()
    ~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 86, in run
    return asyncio_run(self.serve(so

🚀 API server running on http://localhost:8000

Available endpoints:
  GET  /health
  GET  /api/v1/creators?niche=VFX&status=DRAFT_STAGED&min_score=80
  GET  /api/v1/creators/{id}
  GET  /api/v1/creators/{id}/snapshots
  GET  /api/v1/creators/{id}/outreach
  GET  /api/v1/stats/pipeline
  GET  /api/v1/stats/niches
  GET  /api/v1/search?q=studio
  POST /api/v1/sync/trigger?max_creators=100


### 5.2 · Test API Endpoints


In [16]:
import httpx

base = "http://localhost:8000"

print("🧪 Testing API endpoints...\n")

# Health check
r = httpx.get(f"{base}/health")
print(f"GET /health → {r.status_code}")
print(f"   {r.json()}\n")

# Pipeline stats
r = httpx.get(f"{base}/api/v1/stats/pipeline")
data = r.json()
print(f"GET /api/v1/stats/pipeline → {r.status_code}")
print(f"   Total creators: {data['total_creators']}")
for s in data["by_status"][:5]:
    print(f"   {s['status']:25s} {s['count']:5d} ({s['pct']}%)")
print()

# List creators with filters
r = httpx.get(f"{base}/api/v1/creators?niche=VFX&min_score=80&page_size=5")
data = r.json()
print(f"GET /api/v1/creators?niche=VFX&min_score=80 → {r.status_code}")
print(f"   {data['total']} matches, showing {len(data['creators'])}:")
for c in data["creators"]:
    print(f"   @{c['handle']:30s} score={c['total_score']} ER={c['engagement_rate']:.2f}%")
print()

# Get single creator
r = httpx.get(f"{base}/api/v1/creators/1")
data = r.json()
print(f"GET /api/v1/creators/1 → {r.status_code}")
print(f"   @{data['handle']}: {data['followers']:,} followers, "
      f"ER={data['engagement_rate']:.2f}%")
print(f"   Emails: {data.get('emails', [])}")
print()

# Niches
r = httpx.get(f"{base}/api/v1/stats/niches")
data = r.json()
print(f"GET /api/v1/stats/niches → {r.status_code}")
for n in data["niches"][:5]:
    print(f"   {n['niche']:25s} {n['count']:4d} creators  "
          f"avg {n['avg_followers']:>8,} followers  ER {n['avg_er']:.2f}%")
print()

# Search
r = httpx.get(f"{base}/api/v1/search?q=art&limit=5")
data = r.json()
print(f"GET /api/v1/search?q=art → {r.status_code}")
print(f"   {data['count']} results:")
for c in data["results"][:3]:
    print(f"   @{c['handle']:30s} score={c['score']} status={c['status']}")

# Engagement snapshots
r = httpx.get(f"{base}/api/v1/creators/1/snapshots?limit=5")
data = r.json()
print(f"\nGET /api/v1/creators/1/snapshots → {r.status_code}")
print(f"   @{data['handle']} — {len(data['snapshots'])} snapshots:")
for s in data["snapshots"][:3]:
    print(f"   {s['date'][:10]}  followers={s['followers']:,}  ER={s['engagement_rate']:.2f}%")

print(f"\n✅ All endpoints tested successfully")



🧪 Testing API endpoints...



ConnectError: [Errno 111] Connection refused

---
## 6 · Meta Graph API — Complete Authentication Guide

### Step-by-Step Setup for Production


In [ ]:
guide = """
╔══════════════════════════════════════════════════════════════════════╗
║           META GRAPH API — AUTHENTICATION & SETUP GUIDE            ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

STEP 1: CREATE A META DEVELOPER APP
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  → https://developers.facebook.com/apps/create/
  → Select "Business" app type
  → Name: "Shadow-Operation"
  → Link your Business Manager account

STEP 2: ADD INSTAGRAM GRAPH API
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  → App Dashboard → Add Products → Instagram Graph API → Set Up

STEP 3: CONFIGURE APP
━━━━━━━━━━━━━━━━━━━━
  → Settings → Basic:
      • Note your App ID (META_APP_ID)
      • Note your App Secret (META_APP_SECRET)
      • Add App Domains
      • Set Privacy Policy URL + Terms of Service URL

  → Settings → Advanced → Security:
      • Enable "Require App Secret"
      • Set "Allow API Access to App Settings" = Yes

  → Facebook Login → Settings:
      • Add Valid OAuth Redirect URIs:
        https://yourdomain.com/auth/meta/callback

STEP 4: REQUEST ADVANCED ACCESS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Standard Access is limited to 25 accounts.
  For full platform, request Advanced Access:

  ┌─────────────────────────────────┬───────────────────────────────┐
  │ Permission                      │ What It Unlocks               │
  ├─────────────────────────────────┼───────────────────────────────┤
  │ instagram_basic                 │ Profile: username, bio,       │
  │                                 │ followers, media count        │
  ├─────────────────────────────────┼───────────────────────────────┤
  │ instagram_manage_insights       │ Per-post engagement:          │
  │                                 │ likes, comments, reach, saves │
  ├─────────────────────────────────┼───────────────────────────────┤
  │ pages_show_list                 │ List Pages → resolve          │
  │                                 │ Instagram User IDs            │
  ├─────────────────────────────────┼───────────────────────────────┤
  │ pages_read_engagement           │ Page content + engagement     │
  └─────────────────────────────────┴───────────────────────────────┘

  To get approved:
  → App Review → Permissions and Features → Request Advanced Access
  → Describe your use case + screencast
  → Show your privacy policy
  → Meta responds in 3-5 business days

STEP 5: AUTHENTICATION (3 OPTIONS)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  OPTION A: System User Token (RECOMMENDED for production)
  ────────────────────────────────────────────────────────
  Best for: server-side, never expires, highest rate limits

  → Business Manager → Business Settings → Users → System Users
  → Create System User with Admin role
  → Assign assets: Instagram accounts + Facebook Pages
  → Generate token with scopes:
      instagram_basic
      instagram_manage_insights
      pages_show_list
      pages_read_engagement

  TOKEN NEVER EXPIRES. Store in environment variable.

  OPTION B: OAuth 2.0 Flow (for user-facing features)
  ──────────────────────────────────────────────────
  Best for: creators connecting their own accounts

  Step 5B.1: Redirect user to:
    https://www.facebook.com/v21.0/dialog/oauth?
      client_id={app-id}
      &redirect_uri={redirect-uri}
      &scope=instagram_basic,instagram_manage_insights,pages_show_list
      &response_type=code

  Step 5B.2: Exchange code → short-lived token (1-2 hours):
    GET https://graph.facebook.com/v21.0/oauth/access_token?
      client_id={app-id}
      &redirect_uri={redirect-uri}
      &client_secret={app-secret}
      &code={code}

  Step 5B.3: Exchange → long-lived token (~60 days):
    GET https://graph.facebook.com/v21.0/oauth/access_token?
      grant_type=fb_exchange_token
      &client_id={app-id}
      &client_secret={app-secret}
      &fb_exchange_token={short-lived-token}

  Step 5B.4: Refresh before expiry (within 60 days):
    Same endpoint as 5B.3, pass current long-lived token

  OPTION C: Page Access Token
  ───────────────────────────
  Required for some Page-level operations.

  GET /me/accounts?fields=access_token,name
  → Returns Page tokens for each Page you manage

STEP 6: RESOLVE INSTAGRAM USERNAME → IG USER ID
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  The IG User ID is required for ALL subsequent API calls.

  GET /me/accounts?fields=id,name,instagram_business_account
  → Returns: { "data": [{ "instagram_business_account": { "id": "178414..." } }] }

  If you already have the IG User ID:
  GET /{ig-user-id}?fields=id,username

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

API ENDPOINTS MAP
━━━━━━━━━━━━━━━━━

  ┌──────────────────────────────────┬────────┬───────────────────────────┐
  │ Endpoint                         │ Method │ Purpose                   │
  ├──────────────────────────────────┼────────┼───────────────────────────┤
  │ /{ig-user-id}                    │ GET    │ Profile (followers, bio)  │
  │ /{ig-user-id}/media              │ GET    │ Recent posts              │
  │ /{media-id}/insights             │ GET    │ Per-post engagement       │
  │ /{ig-user-id}/insights           │ GET    │ Account-level metrics     │
  │ /me/accounts                     │ GET    │ Pages → IG User IDs       │
  │ / (batch)                        │ POST   │ 50 requests per call      │
  │ /oauth/access_token              │ GET    │ Token exchange/refresh    │
  │ /debug_token                     │ GET    │ Inspect token metadata    │
  └──────────────────────────────────┴────────┴───────────────────────────┘

  Batch API (most efficient):
    POST /
    body: batch=[{method:"GET", relative_url:"{uid}?fields=followers_count,..."}]
    → 50 profiles in 1 HTTP call

  Rate Limits:
    Standard:  200 calls/user/hour
    Business:  4,800 calls/app/day
    Batch:     50 requests per call = 50x efficiency

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PRODUCTION CHECKLIST
━━━━━━━━━━━━━━━━━━━━
  [ ] App Review submitted + Advanced Access approved
  [ ] System User token created (not short-lived OAuth)
  [ ] Token stored in env var / secrets manager
  [ ] Rate limiter: sliding window 180/hr
  [ ] Batch API for bulk (>10 creators)
  [ ] Exponential backoff on 429 / 5xx
  [ ] Webhook endpoint for real-time updates
  [ ] Error logging captures fbtrace_id
  [ ] GDPR: data deletion endpoint
  [ ] Privacy policy updated
  [ ] Monitoring: API calls, error rate, sync lag

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR CODES
━━━━━━━━━━━
  Code 1:   API Unknown         → Retry with backoff
  Code 4:   Too Many Calls      → Rate limit, wait + retry
  Code 10:  Permission Denied   → Check scopes / App Review
  Code 17:  User Too Many Calls → Per-user rate limit
  Code 100: Invalid Params      → Fix request parameters
  Code 190: Invalid OAuth Token → Refresh or re-authenticate
  Code 2500: Method Unsupported → Wrong endpoint or version

  When filing Meta bugs, collect:
    • Error code + message
    • fbtrace_id from the error
    • Exact request URL + params
    • App ID (never share App Secret)
    → Submit: https://developers.facebook.com/support/
"""

print(guide)



---
## 7 · Architecture Summary


In [ ]:
architecture = """
┌─────────────────────────────────────────────────────────────────┐
│                  SHADOW-OPERATION BACKEND                       │
│                  (Running in Google Colab)                      │
│                                                                 │
│  ┌──────────────┐  ┌──────────────────┐  ┌──────────────────┐ │
│  │ FastAPI       │  │ MetaGraphClient   │  │ Database         │ │
│  │ REST API      │  │ (async, batch)    │  │ (SQLite/prod:    │ │
│  │ 9 endpoints   │→│ • Rate limiter    │  │  PostgreSQL)     │ │
│  │               │  │ • Retry + backoff │  │                  │ │
│  │ /creators     │  │ • 50/batch call   │  │ 7 tables         │ │
│  │ /stats        │  │ • Token mgmt     │  │ 30+ indexes      │ │
│  │ /search       │  │ • ER computation  │  │ Full-text search │ │
│  │ /sync         │  │                   │  │ Time-series      │ │
│  └──────┬───────┘  └────────┬──────────┘  └────────┬─────────┘ │
│         │                   │                      │           │
│         └───────────────────┼──────────────────────┘           │
│                             │                                   │
│                    ┌────────▼──────────┐                       │
│                    │ BulkUpdater       │                       │
│                    │ • Select stale    │                       │
│                    │ • Batch fetch     │                       │
│                    │ • Apply updates   │                       │
│                    │ • Log snapshots   │                       │
│                    │ • Audit sync_log  │                       │
│                    └────────┬──────────┘                       │
│                             │                                   │
│                    ┌────────▼──────────┐                       │
│                    │ Meta Graph API    │                       │
│                    │ v21.0             │                       │
│                    └───────────────────┘                       │
└─────────────────────────────────────────────────────────────────┘

QUERY PATTERNS THE INDEXES OPTIMIZE:
  • handle → unique index (O(log n) lookup)
  • niche + status + total_score → composite index
  • followers 5K-500K → partial index (only qualifying rows)
  • engagement_rate > 0 → partial index
  • handle + bio → GIN full-text search
  • ethnicity + niche → composite for demographic filtering
  • location_country + region → geographic queries
  • campaign + status → pipeline by campaign
"""
print(architecture)



---
## 8 · Export & Next Steps

### Database file location
The SQLite database is at `/content/shadow_op.db`.
Download it or mount Google Drive to persist.


In [ ]:
# ── Database Stats ──────────────────────────────────────────────────────
session = SessionLocal()
total_creators = session.query(func.count(Creator.id)).scalar()
total_contacts = session.query(func.count(CreatorContact.id)).scalar()
total_snapshots = session.query(func.count(EngagementSnapshot.id)).scalar()
total_campaigns = session.query(func.count(Campaign.id)).scalar()

print("📊 Final Database State:")
print(f"   Creators:              {total_creators:,}")
print(f"   Contacts (emails):     {total_contacts:,}")
print(f"   Engagement snapshots:  {total_snapshots:,}")
print(f"   Campaigns:             {total_campaigns}")

# ── File size ────────────────────────────────────────────────────────────
import os
size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f"   Database size:         {size_mb:.2f} MB")
print(f"   Location:              {DB_PATH}")

session.close()

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

NEXT STEPS
━━━━━━━━━━
1. DOWNLOAD THE DATABASE
   from google.colab import files
   files.download('/content/shadow_op.db')

2. SET META CREDENTIALS
   os.environ["META_ACCESS_TOKEN"] = "your_system_user_token"
   os.environ["META_APP_ID"] = "your_app_id"
   os.environ["META_APP_SECRET"] = "your_app_secret"

3. RUN LIVE SYNC
   client = MetaGraphClient()
   profile = await client.get_user_profile("17841400123456789")

4. DEPLOY TO PRODUCTION
   • Replace SQLite with PostgreSQL (change engine URL)
   • Run Alembic migrations: alembic upgrade head
   • Deploy FastAPI: uvicorn backend.api.main:app
   • Start Celery: celery -A backend.services.tasks worker
   • Set up monitoring on /health endpoint

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


---
**Built by Shadow-Operation • Backend Infrastructure v1.0**
